# 🚀 Notebook 3A — Maximum DNABERT Fine-Tuning on Perlmutter

## One expensive model. Use the GPUs to train it hard.

Notebook 3 asked:

> **Does adding GPUs make the same workload faster?**

Notebook 3A changes the goal.

We now want to build **one serious DNABERT classifier** and use the GPU allocation as effectively as we reasonably can.

### Main idea

```text
full DNABERT encoder
+ classification head
+ every useful parameter trainable
+ all requested GPUs
+ DDP
+ BF16 mixed precision
+ large per-GPU batch
+ fused optimizer when available
        ↓
maximum useful fine-tuning workload
```

This is **not** a fair 1-GPU vs 2-GPU scaling experiment.

Notebook 3 already taught that.

Notebook 3A is a **throughput-first capstone**.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain what **full fine-tuning** means.
2. Verify that essentially **100% of the useful model parameters** receive gradients.
3. Use multiple A100 GPUs on one DNABERT training job with DDP.
4. Use BF16 automatic mixed precision to increase GPU throughput.
5. Choose a **local batch size** that makes better use of GPU memory.
6. Measure:
   - trainable parameters,
   - GPU memory usage,
   - examples per second,
   - training time,
   - validation AUROC/AUPRC.
7. Save a trained DNABERT checkpoint for later use.

# 1. What Does “Maximum Fine-Tuning” Mean?

There are two very different ideas:

### Parameter-efficient fine-tuning

```text
freeze most of DNABERT
↓
train only a small subset
```

Examples include training only the classifier or only the last few layers.

### Full fine-tuning

```text
embeddings
+
all Transformer encoder layers
+
classification head
        ↓
all optimized together
```

Notebook 3A uses **full fine-tuning**.

The BERT pooler is not included because our classifier does not use `pooler_output`.
Instead, we directly classify the final hidden state of the `[CLS]` token.

That means the model contains **only parameters that participate in the task**.

## Why not increase `max_length` to 512?

Our DNA sequences are about **200 bp**.

With overlapping 6-mers:

```text
200 bp
↓
195 overlapping 6-mers
↓
special tokens
↓
fits inside max_length = 256
```

Using 512 would mostly add padding.

That would consume more memory without adding biological information.

So we spend our GPU budget on:

```text
more trainable parameters
larger batches
more epochs
multiple GPUs
mixed precision
```

—not unnecessary padding.

# 2. Setup

In [ ]:
import os
import sys
import json
import time
import shlex
import subprocess
import py_compile

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path(
    os.path.expanduser("~")
)

DATA_DIR = (
    PROJECT_DIR
    / "ctcf_k562_example"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "notebook3a_results"
)

SCRIPTS_DIR = (
    PROJECT_DIR
    / "notebook3a_scripts"
)

SLURM_LOG_DIR = (
    RESULTS_DIR
    / "slurm_logs"
)

for directory in [
    RESULTS_DIR,
    SCRIPTS_DIR,
    SLURM_LOG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ---------------------------------------------------------
# IMPORTANT:
# Do not assume the current Jupyter kernel is the same
# Python environment that should run the Slurm training job.
#
# Test likely dna-llm interpreters and choose the first one
# that can import the packages required by Notebook 3A.
# ---------------------------------------------------------

PYTHON_CANDIDATES = [
    # Shared project environment — best for a bootcamp if available.
    Path(
        "/global/cfs/cdirs/m4388/envs/dna-llm/bin/python"
    ),

    # User-local environment used in earlier Notebook 3 runs.
    Path.home()
    / ".conda"
    / "envs"
    / "dna-llm"
    / "bin"
    / "python",

    # Current Jupyter kernel as a final candidate.
    Path(
        sys.executable
    ),
]


def python_environment_check(
    python_path,
):
    """
    Return (works, message).

    A usable training interpreter must import all packages
    needed by the DNABERT Slurm program.
    """

    if not python_path.exists():
        return (
            False,
            "path does not exist",
        )

    command = [
        str(python_path),
        "-c",
        (
            "import sys; "
            "import torch; "
            "import transformers; "
            "import sklearn; "
            "import pandas; "
            "import numpy; "
            "print(sys.executable); "
            "print('torch=' + torch.__version__); "
            "print('transformers=' + transformers.__version__)"
        ),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        message = (
            result.stderr.strip()
            or result.stdout.strip()
            or "import check failed"
        )

        return (
            False,
            message,
        )

    return (
        True,
        result.stdout.strip(),
    )


NOTEBOOK_PYTHON = None

seen = set()

for candidate in PYTHON_CANDIDATES:
    candidate = candidate.resolve()

    if candidate in seen:
        continue

    seen.add(
        candidate
    )

    works, message = (
        python_environment_check(
            candidate
        )
    )

    print()
    print(
        "Python candidate:",
        candidate,
    )

    if works:
        print(
            "✅ Required packages available"
        )

        print(message)

        if NOTEBOOK_PYTHON is None:
            NOTEBOOK_PYTHON = str(
                candidate
            )

    else:
        print(
            "❌ Not usable for DNABERT"
        )

        print(
            message.splitlines()[-1]
            if message
            else "unknown error"
        )


if NOTEBOOK_PYTHON is None:
    raise RuntimeError(
        "Notebook 3A could not find a Python interpreter "
        "that imports torch, transformers, sklearn, pandas, "
        "and numpy. Select/repair the dna-llm environment "
        "before submitting the GPU job."
    )


print()
print(
    "✅ Slurm training Python:",
    NOTEBOOK_PYTHON,
)

print(
    "Data directory:",
    DATA_DIR,
)

print(
    "Results directory:",
    RESULTS_DIR,
)

# 3. Student GPU Configuration

The current testing environment uses the `shared` QOS, so keep the request at **1 or 2 GPUs**.

When the bootcamp allocation/QOS is available, the same notebook can request more GPUs if that allocation permits it.

## Throughput-first batch design

Instead of fixing the **global** batch as we did for the controlled scaling experiment, Notebook 3A fixes a target **local batch per GPU**.

Example:

```text
2 GPUs × local batch 32 = global batch 64
4 GPUs × local batch 32 = global batch 128
```

That intentionally gives every GPU substantial work.

This is a throughput experiment, not a controlled hardware comparison.

In [ ]:
# ✏️ EDIT ME — hardware + maximum fine-tuning configuration

NERSC_ACCOUNT = "m4388"

# Current testing value.
# Replace later if the bootcamp receives a different QOS.
SLURM_QOS = "shared"

# Current shared-QOS demonstration:
GPU_COUNT = 2

# Give EACH GPU a substantial batch.
# If GPU memory remains low, students can try 48 or 64 later.
LOCAL_BATCH_SIZE = 32

GLOBAL_BATCH_SIZE = (
    GPU_COUNT
    * LOCAL_BATCH_SIZE
)

MAX_CONFIG = {
    "run_name": "dnabert_maximum_full_finetune",
    "gpus": GPU_COUNT,

    # More training than our short demonstration runs.
    "epochs": 10,

    # Derived so every GPU gets LOCAL_BATCH_SIZE examples.
    "batch_size": GLOBAL_BATCH_SIZE,

    # Full BERT fine-tuning usually needs a smaller LR
    # than a randomly initialized classifier.
    "learning_rate": 2e-5,

    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "dropout": 0.10,

    # 200-bp DNA already fits with 6-mer tokenization.
    "max_length": 256,

    # A100-oriented precision mode.
    "precision": "bf16",

    "seed": 42,
}

pd.Series(
    MAX_CONFIG,
    name="value",
)

In [ ]:
# 🔒 RUN ONLY — configuration safety checks

if MAX_CONFIG["gpus"] < 1:
    raise ValueError(
        "Request at least one GPU."
    )

if (
    SLURM_QOS == "shared"
    and MAX_CONFIG["gpus"] > 2
):
    raise ValueError(
        "Current shared-QOS testing should use "
        "1 or 2 GPUs. Change the QOS when the "
        "bootcamp allocation is available."
    )

if (
    MAX_CONFIG["batch_size"]
    % MAX_CONFIG["gpus"]
    != 0
):
    raise ValueError(
        "Global batch size must be divisible "
        "by GPU count."
    )

print(
    "GPUs:",
    MAX_CONFIG["gpus"],
)

print(
    "Global batch:",
    MAX_CONFIG["batch_size"],
)

print(
    "Local batch per GPU:",
    MAX_CONFIG["batch_size"]
    // MAX_CONFIG["gpus"],
)

# 4. How Notebook 3A Uses the GPU

## DDP

Every GPU receives a complete copy of DNABERT.

```text
GPU 0 / rank 0 ─┐
                 ├─ synchronize gradients ─→ one model
GPU 1 / rank 1 ─┘
```

`DistributedSampler` gives different training examples to each rank.

After backward propagation, DDP uses an **all-reduce** so the replicas receive synchronized gradients.

---

## BF16 mixed precision

Model weights remain in normal training precision, while compatible forward operations are automatically executed in BF16.

Conceptually:

```text
FP32 model parameters
        ↓
autocast
        ↓
many matrix operations use BF16
        ↓
less memory + faster tensor-core-friendly computation
        ↓
backward + optimizer
```

We do **not** manually convert the whole model to BF16.

---

## DDP memory optimization

The DDP wrapper uses:

```python
gradient_as_bucket_view=True
```

so gradient tensors can share storage with DDP communication buckets after the first iteration.

The goal is to reduce unnecessary gradient-memory duplication.

# 5. Parameter Goal

Notebook 3A removes BERT's unused pooling layer at model construction:

```python
BertModel.from_pretrained(
    MODEL_NAME,
    add_pooling_layer=False,
)
```

Then every remaining parameter is trainable.

The audit we want is:

```text
useful trainable parameters
---------------------------
total model parameters

≈ 100%
```

If that value is below 100%, students should investigate why.

# 6. Write the DDP Helper Module

In [ ]:
# 🔒 RUN ONLY — write ddp_common.py

DDP_COMMON_SCRIPT = (
    SCRIPTS_DIR
    / "ddp_common.py"
)

DDP_COMMON_SOURCE = '\nimport json\nimport os\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\n\nfrom sklearn.metrics import (\n    roc_auc_score,\n    average_precision_score,\n    confusion_matrix,\n    precision_score,\n    recall_score,\n    f1_score,\n)\n\n\ndef setup_distributed():\n    rank = int(\n        os.environ.get(\n            "RANK",\n            "0",\n        )\n    )\n\n    local_rank = int(\n        os.environ.get(\n            "LOCAL_RANK",\n            "0",\n        )\n    )\n\n    world_size = int(\n        os.environ.get(\n            "WORLD_SIZE",\n            "1",\n        )\n    )\n\n    distributed = (\n        world_size > 1\n    )\n\n    visible_gpu_count = (\n        torch.cuda.device_count()\n    )\n\n    cuda_visible_devices = (\n        os.environ.get(\n            "CUDA_VISIBLE_DEVICES",\n            "<not set>",\n        )\n    )\n\n    print(\n        f"[DDP setup] "\n        f"RANK={rank} | "\n        f"LOCAL_RANK={local_rank} | "\n        f"WORLD_SIZE={world_size} | "\n        f"visible_GPUs={visible_gpu_count} | "\n        f"CUDA_VISIBLE_DEVICES="\n        f"{cuda_visible_devices}",\n        flush=True,\n    )\n\n    if visible_gpu_count < world_size:\n        raise RuntimeError(\n            f"Rank {rank} sees only "\n            f"{visible_gpu_count} GPU(s), "\n            f"but WORLD_SIZE={world_size}."\n        )\n\n    if local_rank >= visible_gpu_count:\n        raise RuntimeError(\n            f"LOCAL_RANK={local_rank}, but only "\n            f"{visible_gpu_count} GPU ordinal(s) "\n            "are visible."\n        )\n\n    torch.cuda.set_device(\n        local_rank\n    )\n\n    device = torch.device(\n        "cuda",\n        local_rank,\n    )\n\n    print(\n        f"[GPU mapping] "\n        f"rank={rank} → cuda:{local_rank} | "\n        f"{torch.cuda.get_device_name(local_rank)}",\n        flush=True,\n    )\n\n    if distributed:\n        dist.init_process_group(\n            backend="nccl",\n            init_method="env://",\n            rank=rank,\n            world_size=world_size,\n        )\n\n    return (\n        distributed,\n        rank,\n        world_size,\n        local_rank,\n        device,\n    )\n\n\ndef cleanup_distributed(distributed):\n    if (\n        distributed\n        and dist.is_initialized()\n    ):\n        dist.destroy_process_group()\n\n\ndef reduce_training_stats(\n    loss_sum,\n    correct,\n    n,\n    device,\n    distributed,\n):\n    values = torch.tensor(\n        [\n            loss_sum,\n            correct,\n            n,\n        ],\n        dtype=torch.float64,\n        device=device,\n    )\n\n    if distributed:\n        dist.all_reduce(\n            values,\n            op=dist.ReduceOp.SUM,\n        )\n\n    loss_total, correct_total, n_total = (\n        values.tolist()\n    )\n\n    return (\n        loss_total / n_total,\n        correct_total / n_total,\n        int(n_total),\n    )\n\n\ndef binary_metrics(\n    y_true,\n    y_pred,\n    scores,\n):\n    tn, fp, fn, tp = (\n        confusion_matrix(\n            y_true,\n            y_pred,\n            labels=[0, 1],\n        ).ravel()\n    )\n\n    return {\n        "accuracy": (\n            (tp + tn)\n            / (tp + tn + fp + fn)\n        ),\n        "precision": precision_score(\n            y_true,\n            y_pred,\n            zero_division=0,\n        ),\n        "recall": recall_score(\n            y_true,\n            y_pred,\n            zero_division=0,\n        ),\n        "specificity": (\n            tn / (tn + fp)\n            if (tn + fp)\n            else np.nan\n        ),\n        "f1": f1_score(\n            y_true,\n            y_pred,\n            zero_division=0,\n        ),\n        "auroc": roc_auc_score(\n            y_true,\n            scores,\n        ),\n        "auprc": average_precision_score(\n            y_true,\n            scores,\n        ),\n        "true_negative": int(tn),\n        "false_positive": int(fp),\n        "false_negative": int(fn),\n        "true_positive": int(tp),\n    }\n'

DDP_COMMON_SCRIPT.write_text(
    DDP_COMMON_SOURCE
)

py_compile.compile(
    str(DDP_COMMON_SCRIPT),
    doraise=True,
)

print(
    "✅ Wrote:",
    DDP_COMMON_SCRIPT,
)

# 7. Build the Maximum DNABERT Training Program

In [ ]:
# 🔒 RUN ONLY — write maximum_dnabert.py
MAX_SCRIPT = (
    SCRIPTS_DIR
    / "maximum_dnabert.py"
)

MAX_SCRIPT_SOURCE = '\nimport argparse\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom torch.nn.parallel import (\n    DistributedDataParallel as DDP,\n)\n\nfrom torch.utils.data import (\n    Dataset,\n    DataLoader,\n)\n\nfrom torch.utils.data.distributed import (\n    DistributedSampler,\n)\n\nfrom sklearn.model_selection import (\n    train_test_split,\n)\n\nfrom sklearn.metrics import (\n    roc_auc_score,\n    average_precision_score,\n)\n\nfrom transformers import (\n    AutoTokenizer,\n    BertModel,\n    get_linear_schedule_with_warmup,\n)\n\nfrom ddp_common import (\n    setup_distributed,\n    cleanup_distributed,\n    reduce_training_stats,\n    binary_metrics,\n)\n\n\nMODEL_NAME = (\n    "zhihan1996/DNA_bert_6"\n)\n\n\ndef clean_split(\n    data_dir,\n    seed,\n):\n    data_dir = Path(\n        data_dir\n    )\n\n    seqs = [\n        line.strip().upper()\n        for line in open(\n            data_dir / "seqs.txt"\n        )\n        if line.strip()\n    ]\n\n    labels = [\n        int(line.strip())\n        for line in open(\n            data_dir / "labels.txt"\n        )\n        if line.strip()\n    ]\n\n    df = pd.DataFrame({\n        "sequence": seqs,\n        "label": labels,\n    })\n\n    df["length"] = (\n        df["sequence"].str.len()\n    )\n\n    expected_length = int(\n        df["length"].mode().iloc[0]\n    )\n\n    valid = (\n        df["length"].eq(\n            expected_length\n        )\n        & df["sequence"].apply(\n            lambda seq:\n            set(seq)\n            <= set("ACGT")\n        )\n    )\n\n    conflicts = set(\n        df.groupby(\n            "sequence"\n        )["label"]\n        .nunique()\n        .loc[\n            lambda values:\n            values > 1\n        ]\n        .index\n    )\n\n    clean_df = (\n        df[\n            valid\n            & ~df[\n                "sequence"\n            ].isin(\n                conflicts\n            )\n        ]\n        .drop_duplicates(\n            "sequence"\n        )\n        .reset_index(\n            drop=True\n        )\n    )\n\n    train_df, val_df = (\n        train_test_split(\n            clean_df,\n            test_size=0.20,\n            random_state=seed,\n            stratify=(\n                clean_df[\n                    "label"\n                ]\n            ),\n        )\n    )\n\n    return (\n        train_df.reset_index(\n            drop=True\n        ),\n        val_df.reset_index(\n            drop=True\n        ),\n    )\n\n\ndef kmer_sentence(\n    seq,\n    k=6,\n):\n    return " ".join(\n        seq[i:i+k]\n        for i in range(\n            len(seq) - k + 1\n        )\n    )\n\n\nclass DNASet(Dataset):\n    def __init__(\n        self,\n        df,\n        tokenizer,\n        max_length,\n    ):\n        sentences = [\n            kmer_sentence(seq)\n            for seq in df[\n                "sequence"\n            ]\n        ]\n\n        encoded = tokenizer(\n            sentences,\n            padding="max_length",\n            truncation=True,\n            max_length=max_length,\n            return_tensors="pt",\n        )\n\n        self.inputs = (\n            encoded["input_ids"]\n        )\n\n        self.masks = (\n            encoded[\n                "attention_mask"\n            ]\n        )\n\n        self.labels = torch.tensor(\n            df[\n                "label"\n            ].to_numpy(),\n            dtype=torch.long,\n        )\n\n    def __len__(self):\n        return len(\n            self.labels\n        )\n\n    def __getitem__(\n        self,\n        idx,\n    ):\n        return {\n            "input":\n            self.inputs[idx],\n\n            "attention_mask":\n            self.masks[idx],\n\n            "label":\n            self.labels[idx],\n        }\n\n\nclass MaximumDNABertClassifier(\n    nn.Module\n):\n    def __init__(\n        self,\n        dropout,\n    ):\n        super().__init__()\n\n        # IMPORTANT:\n        # We remove the BERT pooler entirely because\n        # this classifier directly uses the final\n        # hidden state of the [CLS] token.\n        self.bert = (\n            BertModel.from_pretrained(\n                MODEL_NAME,\n                add_pooling_layer=False,\n            )\n        )\n\n        hidden_size = (\n            self.bert.config.hidden_size\n        )\n\n        self.dropout = nn.Dropout(\n            dropout\n        )\n\n        self.classifier = nn.Linear(\n            hidden_size,\n            2,\n        )\n\n        # FULL FINE-TUNING:\n        # every remaining parameter receives gradients.\n        for parameter in (\n            self.parameters()\n        ):\n            parameter.requires_grad = True\n\n    def forward(\n        self,\n        x,\n        attention_mask,\n    ):\n        output = self.bert(\n            input_ids=x,\n            attention_mask=(\n                attention_mask\n            ),\n        )\n\n        cls_vector = (\n            output\n            .last_hidden_state[\n                :, 0, :\n            ]\n        )\n\n        return self.classifier(\n            self.dropout(\n                cls_vector\n            )\n        )\n\n\ndef parameter_group_table(\n    model,\n):\n    rows = []\n\n    for name, parameter in (\n        model.named_parameters()\n    ):\n        if name.startswith(\n            "bert.embeddings"\n        ):\n            group = (\n                "BERT embeddings"\n            )\n\n        elif (\n            "bert.encoder.layer."\n            in name\n        ):\n            layer_number = (\n                name.split(\n                    "bert.encoder.layer."\n                )[1]\n                .split(".")[0]\n            )\n\n            group = (\n                "BERT layer "\n                + layer_number\n            )\n\n        elif name.startswith(\n            "classifier"\n        ):\n            group = (\n                "classification head"\n            )\n\n        else:\n            group = "other"\n\n        rows.append({\n            "parameter": name,\n            "group": group,\n            "numel": (\n                parameter.numel()\n            ),\n            "trainable": bool(\n                parameter.requires_grad\n            ),\n        })\n\n    detail = pd.DataFrame(\n        rows\n    )\n\n    grouped = (\n        detail.groupby(\n            [\n                "group",\n                "trainable",\n            ],\n            as_index=False,\n        )["numel"]\n        .sum()\n    )\n\n    return (\n        detail,\n        grouped,\n    )\n\n\ndef build_optimizer(\n    model,\n    learning_rate,\n    weight_decay,\n):\n    no_decay_terms = (\n        "bias",\n        "LayerNorm.weight",\n    )\n\n    decay_parameters = []\n    no_decay_parameters = []\n\n    for name, parameter in (\n        model.named_parameters()\n    ):\n        if any(\n            term in name\n            for term\n            in no_decay_terms\n        ):\n            no_decay_parameters.append(\n                parameter\n            )\n        else:\n            decay_parameters.append(\n                parameter\n            )\n\n    groups = [\n        {\n            "params":\n            decay_parameters,\n\n            "weight_decay":\n            weight_decay,\n        },\n        {\n            "params":\n            no_decay_parameters,\n\n            "weight_decay":\n            0.0,\n        },\n    ]\n\n    try:\n        optimizer = (\n            torch.optim.AdamW(\n                groups,\n                lr=learning_rate,\n                fused=True,\n            )\n        )\n\n        optimizer_mode = (\n            "AdamW fused=True"\n        )\n\n    except (\n        TypeError,\n        RuntimeError,\n    ):\n        optimizer = (\n            torch.optim.AdamW(\n                groups,\n                lr=learning_rate,\n            )\n        )\n\n        optimizer_mode = (\n            "AdamW standard"\n        )\n\n    return (\n        optimizer,\n        optimizer_mode,\n    )\n\n\ndef train_epoch(\n    model,\n    optimizer,\n    scheduler,\n    loader,\n    sampler,\n    epoch,\n    device,\n    distributed,\n):\n    model.train()\n\n    if sampler is not None:\n        sampler.set_epoch(\n            epoch\n        )\n\n    loss_sum = 0.0\n    correct = 0\n    n = 0\n\n    for batch_index, batch in enumerate(\n        loader\n    ):\n        x = batch[\n            "input"\n        ].to(\n            device,\n            non_blocking=True,\n        )\n\n        mask = batch[\n            "attention_mask"\n        ].to(\n            device,\n            non_blocking=True,\n        )\n\n        labels = batch[\n            "label"\n        ].to(\n            device,\n            non_blocking=True,\n        )\n\n        optimizer.zero_grad(\n            set_to_none=True\n        )\n\n        with torch.amp.autocast(\n            device_type="cuda",\n            dtype=torch.bfloat16,\n        ):\n            logits = model(\n                x,\n                mask,\n            )\n\n            loss = (\n                F.cross_entropy(\n                    logits,\n                    labels,\n                )\n            )\n\n        loss.backward()\n\n        # Full-fine-tuning audit:\n        # on the first batch every trainable parameter\n        # should have a gradient.\n        if (\n            epoch == 1\n            and batch_index == 0\n        ):\n            missing_grads = [\n                name\n                for name, parameter\n                in model.named_parameters()\n                if (\n                    parameter.requires_grad\n                    and parameter.grad\n                    is None\n                )\n            ]\n\n            if missing_grads:\n                raise RuntimeError(\n                    "Trainable parameters "\n                    "without gradients: "\n                    + ", ".join(\n                        missing_grads\n                    )\n                )\n\n        torch.nn.utils.clip_grad_norm_(\n            model.parameters(),\n            max_norm=1.0,\n        )\n\n        optimizer.step()\n        scheduler.step()\n\n        predictions = (\n            logits.argmax(\n                dim=1\n            )\n        )\n\n        loss_sum += (\n            loss.item()\n            * len(labels)\n        )\n\n        correct += (\n            predictions\n            == labels\n        ).sum().item()\n\n        n += len(labels)\n\n    return reduce_training_stats(\n        loss_sum,\n        correct,\n        n,\n        device,\n        distributed,\n    )\n\n\n@torch.no_grad()\ndef evaluate(\n    model,\n    loader,\n    device,\n):\n    model.eval()\n\n    loss_sum = 0.0\n    correct = 0\n    n = 0\n\n    scores = []\n    predictions = []\n    truth = []\n\n    for batch in loader:\n        x = batch[\n            "input"\n        ].to(\n            device,\n            non_blocking=True,\n        )\n\n        mask = batch[\n            "attention_mask"\n        ].to(\n            device,\n            non_blocking=True,\n        )\n\n        labels = batch[\n            "label"\n        ].to(\n            device,\n            non_blocking=True,\n        )\n\n        with torch.amp.autocast(\n            device_type="cuda",\n            dtype=torch.bfloat16,\n        ):\n            logits = model(\n                x,\n                mask,\n            )\n\n            loss = (\n                F.cross_entropy(\n                    logits,\n                    labels,\n                )\n            )\n\n        pred = logits.argmax(\n            dim=1\n        )\n\n        prob = torch.softmax(\n            logits.float(),\n            dim=1,\n        )[:, 1]\n\n        loss_sum += (\n            loss.item()\n            * len(labels)\n        )\n\n        correct += (\n            pred\n            == labels\n        ).sum().item()\n\n        n += len(labels)\n\n        scores.extend(\n            prob.cpu().numpy()\n        )\n\n        predictions.extend(\n            pred.cpu().numpy()\n        )\n\n        truth.extend(\n            labels.cpu().numpy()\n        )\n\n    return {\n        "loss":\n        loss_sum / n,\n\n        "accuracy":\n        correct / n,\n\n        "scores":\n        np.asarray(\n            scores\n        ),\n\n        "predictions":\n        np.asarray(\n            predictions\n        ),\n\n        "true":\n        np.asarray(\n            truth\n        ),\n    }\n\n\ndef main():\n    parser = (\n        argparse.ArgumentParser()\n    )\n\n    parser.add_argument(\n        "--data_dir",\n        required=True,\n    )\n\n    parser.add_argument(\n        "--output_dir",\n        required=True,\n    )\n\n    parser.add_argument(\n        "--run_name",\n        required=True,\n    )\n\n    parser.add_argument(\n        "--epochs",\n        type=int,\n        required=True,\n    )\n\n    parser.add_argument(\n        "--batch_size",\n        type=int,\n        required=True,\n    )\n\n    parser.add_argument(\n        "--learning_rate",\n        type=float,\n        required=True,\n    )\n\n    parser.add_argument(\n        "--weight_decay",\n        type=float,\n        required=True,\n    )\n\n    parser.add_argument(\n        "--warmup_ratio",\n        type=float,\n        required=True,\n    )\n\n    parser.add_argument(\n        "--dropout",\n        type=float,\n        required=True,\n    )\n\n    parser.add_argument(\n        "--max_length",\n        type=int,\n        default=256,\n    )\n\n    parser.add_argument(\n        "--precision",\n        choices=[\n            "bf16",\n        ],\n        default="bf16",\n    )\n\n    parser.add_argument(\n        "--seed",\n        type=int,\n        default=42,\n    )\n\n    args = (\n        parser.parse_args()\n    )\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\n            "CUDA GPU required."\n        )\n\n    if not (\n        torch.cuda.is_bf16_supported()\n    ):\n        raise RuntimeError(\n            "Notebook 3A expects "\n            "BF16-capable GPUs."\n        )\n\n    (\n        distributed,\n        rank,\n        world_size,\n        local_rank,\n        device,\n    ) = setup_distributed()\n\n    try:\n        if (\n            args.batch_size\n            % world_size\n            != 0\n        ):\n            raise ValueError(\n                "Global batch size "\n                "must be divisible "\n                "by GPU count."\n            )\n\n        local_batch_size = (\n            args.batch_size\n            // world_size\n        )\n\n        np.random.seed(\n            args.seed\n        )\n\n        torch.manual_seed(\n            args.seed\n        )\n\n        torch.cuda.manual_seed_all(\n            args.seed\n        )\n\n        # Allow optimized matmul paths for remaining FP32 ops.\n        torch.set_float32_matmul_precision(\n            "high"\n        )\n\n        train_df, val_df = (\n            clean_split(\n                args.data_dir,\n                args.seed,\n            )\n        )\n\n        tokenizer = (\n            AutoTokenizer\n            .from_pretrained(\n                MODEL_NAME\n            )\n        )\n\n        train_dataset = DNASet(\n            train_df,\n            tokenizer,\n            args.max_length,\n        )\n\n        val_dataset = DNASet(\n            val_df,\n            tokenizer,\n            args.max_length,\n        )\n\n        train_sampler = None\n\n        if distributed:\n            train_sampler = (\n                DistributedSampler(\n                    train_dataset,\n                    num_replicas=(\n                        world_size\n                    ),\n                    rank=rank,\n                    shuffle=True,\n                    seed=args.seed,\n                )\n            )\n\n        loader_workers = min(\n            4,\n            max(\n                1,\n                (\n                    int(\n                        __import__(\n                            "os"\n                        ).environ.get(\n                            "SLURM_CPUS_PER_TASK",\n                            "4",\n                        )\n                    )\n                    // 4\n                ),\n            ),\n        )\n\n        train_loader = (\n            DataLoader(\n                train_dataset,\n                batch_size=(\n                    local_batch_size\n                ),\n                shuffle=(\n                    train_sampler\n                    is None\n                ),\n                sampler=(\n                    train_sampler\n                ),\n                pin_memory=True,\n                num_workers=(\n                    loader_workers\n                ),\n                persistent_workers=(\n                    loader_workers > 0\n                ),\n            )\n        )\n\n        # Rank 0 performs validation.\n        val_loader = (\n            DataLoader(\n                val_dataset,\n                batch_size=(\n                    local_batch_size\n                ),\n                shuffle=False,\n                pin_memory=True,\n                num_workers=(\n                    loader_workers\n                ),\n                persistent_workers=(\n                    loader_workers > 0\n                ),\n            )\n        )\n\n        model = (\n            MaximumDNABertClassifier(\n                dropout=(\n                    args.dropout\n                )\n            )\n            .to(device)\n        )\n\n        total_parameters = sum(\n            p.numel()\n            for p\n            in model.parameters()\n        )\n\n        trainable_parameters = sum(\n            p.numel()\n            for p\n            in model.parameters()\n            if p.requires_grad\n        )\n\n        trainable_percent = (\n            100.0\n            * trainable_parameters\n            / total_parameters\n        )\n\n        if (\n            trainable_parameters\n            != total_parameters\n        ):\n            raise RuntimeError(\n                "Maximum fine-tuning "\n                "requires all remaining "\n                "model parameters to "\n                "be trainable."\n            )\n\n        parameter_detail, (\n            parameter_groups\n        ) = parameter_group_table(\n            model\n        )\n\n        if distributed:\n            model = DDP(\n                model,\n                device_ids=[\n                    device.index\n                ],\n                output_device=(\n                    device.index\n                ),\n                gradient_as_bucket_view=True,\n                static_graph=True,\n            )\n\n        (\n            optimizer,\n            optimizer_mode,\n        ) = build_optimizer(\n            model,\n            args.learning_rate,\n            args.weight_decay,\n        )\n\n        total_steps = (\n            len(train_loader)\n            * args.epochs\n        )\n\n        warmup_steps = int(\n            total_steps\n            * args.warmup_ratio\n        )\n\n        scheduler = (\n            get_linear_schedule_with_warmup(\n                optimizer,\n                num_warmup_steps=(\n                    warmup_steps\n                ),\n                num_training_steps=(\n                    total_steps\n                ),\n            )\n        )\n\n        # Different dropout streams after model sync.\n        torch.manual_seed(\n            args.seed + rank\n        )\n\n        torch.cuda.manual_seed_all(\n            args.seed + rank\n        )\n\n        if rank == 0:\n            print(\n                "\\n=== MAXIMUM DNABERT ===",\n                flush=True,\n            )\n\n            print(\n                f"GPUs: {world_size}",\n                flush=True,\n            )\n\n            print(\n                f"Precision: "\n                f"{args.precision}",\n                flush=True,\n            )\n\n            print(\n                f"Global batch: "\n                f"{args.batch_size}",\n                flush=True,\n            )\n\n            print(\n                f"Local batch/GPU: "\n                f"{local_batch_size}",\n                flush=True,\n            )\n\n            print(\n                f"Trainable parameters: "\n                f"{trainable_parameters:,}",\n                flush=True,\n            )\n\n            print(\n                f"Total parameters: "\n                f"{total_parameters:,}",\n                flush=True,\n            )\n\n            print(\n                f"Trainable percent: "\n                f"{trainable_percent:.2f}%",\n                flush=True,\n            )\n\n            print(\n                f"Optimizer: "\n                f"{optimizer_mode}",\n                flush=True,\n            )\n\n            print(\n                f"Steps/epoch/rank: "\n                f"{len(train_loader)}",\n                flush=True,\n            )\n\n        if distributed:\n            dist.barrier()\n\n        torch.cuda.empty_cache()\n\n        torch.cuda.reset_peak_memory_stats(\n            device\n        )\n\n        torch.cuda.synchronize(\n            device\n        )\n\n        training_start = (\n            time.time()\n        )\n\n        history = []\n        final_val = None\n        processed_examples = 0\n\n        for epoch in range(\n            1,\n            args.epochs + 1,\n        ):\n            epoch_start = (\n                time.time()\n            )\n\n            (\n                train_loss,\n                train_accuracy,\n                global_examples_seen,\n            ) = train_epoch(\n                model,\n                optimizer,\n                scheduler,\n                train_loader,\n                train_sampler,\n                epoch,\n                device,\n                distributed,\n            )\n\n            processed_examples += (\n                global_examples_seen\n            )\n\n            if distributed:\n                dist.barrier()\n\n            if rank == 0:\n                eval_model = (\n                    model.module\n                    if distributed\n                    else model\n                )\n\n                final_val = evaluate(\n                    eval_model,\n                    val_loader,\n                    device,\n                )\n\n                val_auroc = (\n                    roc_auc_score(\n                        final_val[\n                            "true"\n                        ],\n                        final_val[\n                            "scores"\n                        ],\n                    )\n                )\n\n                val_auprc = (\n                    average_precision_score(\n                        final_val[\n                            "true"\n                        ],\n                        final_val[\n                            "scores"\n                        ],\n                    )\n                )\n\n                epoch_seconds = (\n                    time.time()\n                    - epoch_start\n                )\n\n                row = {\n                    "epoch": epoch,\n                    "train_loss":\n                    train_loss,\n\n                    "train_accuracy":\n                    train_accuracy,\n\n                    "val_loss":\n                    final_val["loss"],\n\n                    "val_accuracy":\n                    final_val[\n                        "accuracy"\n                    ],\n\n                    "val_auroc":\n                    val_auroc,\n\n                    "val_auprc":\n                    val_auprc,\n\n                    "epoch_seconds":\n                    epoch_seconds,\n\n                    "learning_rate":\n                    optimizer.param_groups[\n                        0\n                    ]["lr"],\n                }\n\n                history.append(\n                    row\n                )\n\n                print(\n                    f"Epoch "\n                    f"{epoch}/"\n                    f"{args.epochs} | "\n                    f"train loss="\n                    f"{train_loss:.4f} | "\n                    f"train acc="\n                    f"{train_accuracy:.3f} | "\n                    f"val loss="\n                    f"{final_val[\'loss\']:.4f} | "\n                    f"AUROC="\n                    f"{val_auroc:.4f} | "\n                    f"AUPRC="\n                    f"{val_auprc:.4f} | "\n                    f"{epoch_seconds:.1f}s",\n                    flush=True,\n                )\n\n            if distributed:\n                dist.barrier()\n\n        torch.cuda.synchronize(\n            device\n        )\n\n        training_time = (\n            time.time()\n            - training_start\n        )\n\n        local_peak_bytes = (\n            torch.cuda\n            .max_memory_allocated(\n                device\n            )\n        )\n\n        peak_tensor = torch.tensor(\n            [\n                float(\n                    local_peak_bytes\n                )\n            ],\n            dtype=torch.float64,\n            device=device,\n        )\n\n        if distributed:\n            dist.all_reduce(\n                peak_tensor,\n                op=dist.ReduceOp.MAX,\n            )\n\n        max_peak_bytes = (\n            peak_tensor.item()\n        )\n\n        total_memory_bytes = (\n            torch.cuda\n            .get_device_properties(\n                device\n            )\n            .total_memory\n        )\n\n        peak_memory_gb = (\n            max_peak_bytes\n            / (1024 ** 3)\n        )\n\n        total_memory_gb = (\n            total_memory_bytes\n            / (1024 ** 3)\n        )\n\n        peak_memory_percent = (\n            100.0\n            * max_peak_bytes\n            / total_memory_bytes\n        )\n\n        if rank == 0:\n            history_df = (\n                pd.DataFrame(\n                    history\n                )\n            )\n\n            best_row = (\n                history_df.loc[\n                    history_df[\n                        "val_auroc"\n                    ].idxmax()\n                ]\n            )\n\n            metrics = (\n                binary_metrics(\n                    final_val["true"],\n                    final_val[\n                        "predictions"\n                    ],\n                    final_val["scores"],\n                )\n            )\n\n            predictions_df = (\n                val_df[\n                    [\n                        "sequence",\n                        "label",\n                    ]\n                ]\n                .copy()\n            )\n\n            predictions_df[\n                "predicted_label"\n            ] = (\n                final_val[\n                    "predictions"\n                ]\n            )\n\n            predictions_df[\n                "binding_probability"\n            ] = (\n                final_val[\n                    "scores"\n                ]\n            )\n\n            predictions_df[\n                "correct"\n            ] = (\n                predictions_df[\n                    "label"\n                ]\n                == predictions_df[\n                    "predicted_label"\n                ]\n            )\n\n            # Use the actual number of examples processed by\n            # all DDP ranks. DistributedSampler can pad a small\n            # number of samples to make rank lengths equal.\n            examples_per_second = (\n                processed_examples\n                / training_time\n            )\n\n            output_dir = Path(\n                args.output_dir\n            )\n\n            output_dir.mkdir(\n                parents=True,\n                exist_ok=True,\n            )\n\n            prefix = (\n                output_dir\n                / args.run_name\n            )\n\n            history_df.to_csv(\n                str(prefix)\n                + "_history.csv",\n                index=False,\n            )\n\n            predictions_df.to_csv(\n                str(prefix)\n                + "_predictions.csv",\n                index=False,\n            )\n\n            parameter_detail.to_csv(\n                str(prefix)\n                + "_parameters.csv",\n                index=False,\n            )\n\n            parameter_groups.to_csv(\n                str(prefix)\n                + "_parameter_groups.csv",\n                index=False,\n            )\n\n            eval_model = (\n                model.module\n                if distributed\n                else model\n            )\n\n            # Save after the training timer so checkpoint I/O\n            # does not distort the reported training time.\n            checkpoint_path = (\n                str(prefix)\n                + "_final_checkpoint.pt"\n            )\n\n            torch.save(\n                {\n                    "model_name":\n                    MODEL_NAME,\n\n                    "state_dict":\n                    eval_model.state_dict(),\n\n                    "epoch":\n                    args.epochs,\n\n                    "max_length":\n                    args.max_length,\n\n                    "dropout":\n                    args.dropout,\n                },\n                checkpoint_path,\n            )\n\n            summary = {\n                "family":\n                "dnabert",\n\n                "run_name":\n                args.run_name,\n\n                "model_name":\n                MODEL_NAME,\n\n                "strategy":\n                "maximum_full_finetuning",\n\n                "distributed":\n                bool(distributed),\n\n                "num_gpus":\n                int(world_size),\n\n                "precision":\n                args.precision,\n\n                "epochs":\n                int(args.epochs),\n\n                "global_batch_size":\n                int(args.batch_size),\n\n                "local_batch_size":\n                int(local_batch_size),\n\n                "learning_rate":\n                float(\n                    args.learning_rate\n                ),\n\n                "weight_decay":\n                float(\n                    args.weight_decay\n                ),\n\n                "warmup_ratio":\n                float(\n                    args.warmup_ratio\n                ),\n\n                "dropout":\n                float(args.dropout),\n\n                "max_length":\n                int(args.max_length),\n\n                "random_seed":\n                int(args.seed),\n\n                "optimizer":\n                optimizer_mode,\n\n                "trainable_parameters":\n                int(\n                    trainable_parameters\n                ),\n\n                "total_parameters":\n                int(\n                    total_parameters\n                ),\n\n                "trainable_percent":\n                float(\n                    trainable_percent\n                ),\n\n                "best_val_auroc":\n                float(\n                    best_row[\n                        "val_auroc"\n                    ]\n                ),\n\n                "best_epoch":\n                int(\n                    best_row[\n                        "epoch"\n                    ]\n                ),\n\n                "final_val_accuracy":\n                float(\n                    metrics[\n                        "accuracy"\n                    ]\n                ),\n\n                "final_val_precision":\n                float(\n                    metrics[\n                        "precision"\n                    ]\n                ),\n\n                "final_val_recall":\n                float(\n                    metrics[\n                        "recall"\n                    ]\n                ),\n\n                "final_val_specificity":\n                float(\n                    metrics[\n                        "specificity"\n                    ]\n                ),\n\n                "final_val_f1":\n                float(\n                    metrics[\n                        "f1"\n                    ]\n                ),\n\n                "final_val_auroc":\n                float(\n                    metrics[\n                        "auroc"\n                    ]\n                ),\n\n                "final_val_auprc":\n                float(\n                    metrics[\n                        "auprc"\n                    ]\n                ),\n\n                "training_time_seconds":\n                float(\n                    training_time\n                ),\n\n                "examples_per_second":\n                float(\n                    examples_per_second\n                ),\n\n                "peak_gpu_memory_gb":\n                float(\n                    peak_memory_gb\n                ),\n\n                "gpu_memory_capacity_gb":\n                float(\n                    total_memory_gb\n                ),\n\n                "peak_gpu_memory_percent":\n                float(\n                    peak_memory_percent\n                ),\n\n                "checkpoint_path":\n                checkpoint_path,\n            }\n\n            with open(\n                str(prefix)\n                + "_summary.json",\n                "w",\n            ) as handle:\n                json.dump(\n                    summary,\n                    handle,\n                    indent=2,\n                )\n\n            print()\n            print(\n                "=== FINAL REPORT ===",\n                flush=True,\n            )\n\n            print(\n                f"Best AUROC: "\n                f"{summary[\'best_val_auroc\']:.4f}",\n                flush=True,\n            )\n\n            print(\n                f"Training time: "\n                f"{training_time:.1f}s",\n                flush=True,\n            )\n\n            print(\n                f"Examples/sec: "\n                f"{examples_per_second:.1f}",\n                flush=True,\n            )\n\n            print(\n                f"Peak GPU memory: "\n                f"{peak_memory_gb:.2f} / "\n                f"{total_memory_gb:.2f} GB "\n                f"({peak_memory_percent:.1f}%)",\n                flush=True,\n            )\n\n            print(\n                f"Checkpoint: "\n                f"{checkpoint_path}",\n                flush=True,\n            )\n\n    finally:\n        cleanup_distributed(\n            distributed\n        )\n\n\nif __name__ == "__main__":\n    main()\n'

MAX_SCRIPT.write_text(
    MAX_SCRIPT_SOURCE
)

py_compile.compile(
    str(MAX_SCRIPT),
    doraise=True,
)

print(
    "✅ Python syntax valid:",
    MAX_SCRIPT,
)


## Key code to recognize

Students should be able to explain these four decisions:

### 1. Remove the unused pooler

```python
BertModel.from_pretrained(
    MODEL_NAME,
    add_pooling_layer=False,
)
```

### 2. Train everything that remains

```python
for parameter in self.parameters():
    parameter.requires_grad = True
```

### 3. Use BF16 for compatible forward operations

```python
with torch.amp.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
):
    ...
```

### 4. Use DDP memory-efficient gradient buckets

```python
DDP(
    model,
    ...,
    gradient_as_bucket_view=True,
    static_graph=True,
)
```

# 8. SLURM Submission Helpers

In [ ]:
# 🔒 RUN ONLY — syntax validation + submission monitoring

def validate_shell_script(
    path,
):
    result = subprocess.run(
        [
            "bash",
            "-n",
            str(path),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print(
            result.stderr
        )
        return False

    print(
        "✅ Shell syntax valid:",
        path.name,
    )

    return True


def submit_and_stream(
    script_path,
    job_name,
    poll_seconds=2.0,
):
    # NERSC Jupyter may itself have a CUDA_VISIBLE_DEVICES
    # value. Do not pass that mask into a new batch job.
    submit_env = (
        os.environ.copy()
    )

    inherited_gpu_env = {
        name:
        submit_env.get(name)

        for name in [
            "CUDA_VISIBLE_DEVICES",
            "NVIDIA_VISIBLE_DEVICES",
            "ROCR_VISIBLE_DEVICES",
            "GPU_DEVICE_ORDINAL",
        ]

        if name in submit_env
    }

    for name in [
        "CUDA_VISIBLE_DEVICES",
        "NVIDIA_VISIBLE_DEVICES",
        "ROCR_VISIBLE_DEVICES",
        "GPU_DEVICE_ORDINAL",
    ]:
        submit_env.pop(
            name,
            None,
        )

    print(
        "Notebook GPU environment:",
        inherited_gpu_env
        if inherited_gpu_env
        else "<none>",
    )

    print(
        "Submitting with inherited GPU "
        "visibility removed."
    )

    submit = subprocess.run(
        [
            "sbatch",
            str(script_path),
        ],
        capture_output=True,
        text=True,
        env=submit_env,
    )

    if submit.returncode != 0:
        print(
            submit.stderr
        )

        raise RuntimeError(
            "sbatch rejected the job."
        )

    submit_text = (
        submit.stdout.strip()
    )

    job_id = (
        submit_text.split()[-1]
    )

    output_file = (
        SLURM_LOG_DIR
        / f"{job_name}-{job_id}.out"
    )

    print(
        "Submitted job",
        job_id,
    )

    print(
        "Output:",
        output_file,
    )

    last_size = 0

    while True:
        if output_file.exists():
            with output_file.open(
                "r"
            ) as handle:
                handle.seek(
                    last_size
                )

                text = handle.read()

                if text:
                    print(
                        text,
                        end="",
                    )

                last_size = (
                    handle.tell()
                )

        active = subprocess.run(
            [
                "squeue",
                "-h",
                "-j",
                job_id,
            ],
            capture_output=True,
            text=True,
        ).stdout.strip()

        if not active:
            if output_file.exists():
                with output_file.open(
                    "r"
                ) as handle:
                    handle.seek(
                        last_size
                    )

                    text = (
                        handle.read()
                    )

                    if text:
                        print(
                            text,
                            end="",
                        )

            break

        time.sleep(
            poll_seconds
        )

    summary = subprocess.run(
        [
            "sacct",
            "-j",
            job_id,
            "--format="
            "JobID,State,ExitCode,"
            "Elapsed,AllocTRES",
            "-n",
            "-P",
        ],
        capture_output=True,
        text=True,
    ).stdout.strip()

    print(
        "\n--- sacct summary ---"
    )

    print(summary)

    main_state = None

    for line in (
        summary.splitlines()
    ):
        fields = (
            line.split("|")
        )

        if (
            len(fields) >= 2
            and fields[0]
            == job_id
        ):
            main_state = (
                fields[1]
            )
            break

    if (
        main_state is None
        or not main_state.startswith(
            "COMPLETED"
        )
    ):
        raise RuntimeError(
            f"SLURM job {job_id} "
            f"finished with state "
            f"{main_state}."
        )

    print(
        f"✅ Job {job_id} "
        "completed successfully."
    )

    return job_id

# 9. Build the Maximum-Power Job

## Python environment preflight

Before generating the Slurm job, Notebook 3A verifies the exact Python interpreter that will run on the compute node.

The selected interpreter must successfully import:

```text
torch
transformers
scikit-learn
pandas
numpy
```

This prevents a common HPC problem:

```text
Jupyter kernel Python
        ≠
training-job Python
```

The printed **Slurm training Python** path is the one that will appear in the generated `.slurm` file.

In [ ]:
# 🔒 RUN ONLY — verify the selected Slurm Python one more time

preflight = subprocess.run(
    [
        NOTEBOOK_PYTHON,
        "-c",
        (
            "import sys; "
            "import torch; "
            "import transformers; "
            "print('python:', sys.executable); "
            "print('torch:', torch.__version__); "
            "print('transformers:', transformers.__version__); "
            "print('CUDA build:', torch.version.cuda)"
        ),
    ],
    capture_output=True,
    text=True,
)

print(
    preflight.stdout
)

if preflight.returncode != 0:
    print(
        preflight.stderr
    )

    raise RuntimeError(
        "Selected Slurm Python failed the PyTorch preflight."
    )

print(
    "✅ Python environment preflight passed"
)

In [ ]:
# 🔒 RUN ONLY — build command-line arguments

def build_max_arguments(
    config,
):
    values = [
        "--data_dir",
        str(DATA_DIR),

        "--output_dir",
        str(RESULTS_DIR),

        "--run_name",
        str(
            config[
                "run_name"
            ]
        ),

        "--epochs",
        str(
            config[
                "epochs"
            ]
        ),

        "--batch_size",
        str(
            config[
                "batch_size"
            ]
        ),

        "--learning_rate",
        str(
            config[
                "learning_rate"
            ]
        ),

        "--weight_decay",
        str(
            config[
                "weight_decay"
            ]
        ),

        "--warmup_ratio",
        str(
            config[
                "warmup_ratio"
            ]
        ),

        "--dropout",
        str(
            config[
                "dropout"
            ]
        ),

        "--max_length",
        str(
            config[
                "max_length"
            ]
        ),

        "--precision",
        str(
            config[
                "precision"
            ]
        ),

        "--seed",
        str(
            config[
                "seed"
            ]
        ),
    ]

    return " ".join(
        shlex.quote(value)
        for value in values
    )

In [ ]:
# 🔒 RUN ONLY — generate one DDP SLURM job

def write_max_slurm_job(
    config,
    walltime="00:45:00",
):
    gpu_count = int(
        config["gpus"]
    )

    if (
        SLURM_QOS == "shared"
        and gpu_count > 2
    ):
        raise ValueError(
            "shared testing is limited "
            "to 1 or 2 GPUs here."
        )

    if (
        config["batch_size"]
        % gpu_count
        != 0
    ):
        raise ValueError(
            "Global batch must be "
            "divisible by GPU count."
        )

    arguments = (
        build_max_arguments(
            config
        )
    )

    job_name = (
        "nb3a-"
        + config[
            "run_name"
        ].replace(
            "_",
            "-",
        )
    )[:60]

    path = (
        SCRIPTS_DIR
        / (
            config[
                "run_name"
            ]
            + ".slurm"
        )
    )

    path.write_text(
f"""#!/bin/bash
#SBATCH -A {NERSC_ACCOUNT}
#SBATCH -C gpu
#SBATCH -q {SLURM_QOS}
#SBATCH -t {walltime}

#SBATCH -N 1
#SBATCH --ntasks-per-node={gpu_count}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-node={gpu_count}
#SBATCH --gpu-bind=none

#SBATCH -J {job_name}
#SBATCH -o {SLURM_LOG_DIR}/{job_name}-%j.out

export SLURM_CPU_BIND="cores"

export MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
export MASTER_PORT=$((10000 + SLURM_JOB_ID % 50000))

export NCCL_DEBUG=WARN

echo "[PYTHON] training interpreter: {NOTEBOOK_PYTHON}"

{NOTEBOOK_PYTHON} -c "import sys, torch, transformers; print('[PYTHON]', sys.executable); print('[PYTORCH]', torch.__version__); print('[TRANSFORMERS]', transformers.__version__)"

if [ $? -ne 0 ]; then
    echo "[ERROR] Selected Python cannot import the DNABERT dependencies."
    exit 1
fi

echo "[BATCH] CUDA_VISIBLE_DEVICES before cleanup=${{CUDA_VISIBLE_DEVICES-<unset>}}"
echo "[BATCH] SLURM_JOB_GPUS=${{SLURM_JOB_GPUS-<unset>}}"

# Prevent the Jupyter server's GPU mask from restricting this job.
unset CUDA_VISIBLE_DEVICES
unset NVIDIA_VISIBLE_DEVICES
unset ROCR_VISIBLE_DEVICES
unset GPU_DEVICE_ORDINAL

srun --gpu-bind=none bash -c '
    export RANK=$SLURM_PROCID
    export LOCAL_RANK=$SLURM_LOCALID
    export WORLD_SIZE=$SLURM_NTASKS

    echo "[SLURM→DDP] RANK=$RANK LOCAL_RANK=$LOCAL_RANK WORLD_SIZE=$WORLD_SIZE CUDA_VISIBLE_DEVICES=${{CUDA_VISIBLE_DEVICES-<unset>}}"

    {NOTEBOOK_PYTHON} {MAX_SCRIPT} {arguments}
'
"""
    )

    if not validate_shell_script(
        path
    ):
        raise RuntimeError(
            "Fix shell syntax "
            "before submitting."
        )

    return (
        path,
        job_name,
    )

In [ ]:
# ✏️ RUN THIS — generate and inspect the actual job

MAX_SLURM_SCRIPT, MAX_JOB_NAME = (
    write_max_slurm_job(
        MAX_CONFIG,
        walltime="00:45:00",
    )
)

print(
    MAX_SLURM_SCRIPT.read_text()
)

### ✅ CHECKPOINT — before submitting

For a 2-GPU run, confirm:

```bash
#SBATCH --ntasks-per-node=2
#SBATCH --gpus-per-node=2
#SBATCH --gpu-bind=none
```

and:

```bash
RANK=$SLURM_PROCID
LOCAL_RANK=$SLURM_LOCALID
WORLD_SIZE=$SLURM_NTASKS
```

The training output should later show:

```text
rank 0 → cuda:0
rank 1 → cuda:1
```

# 10. Launch the Maximum DNABERT Run

### Environment checkpoint

Before DDP starts, the Slurm output should print something like:

```text
[PYTHON] /.../dna-llm/bin/python
[PYTORCH] <version>
[TRANSFORMERS] <version>
```

If you instead see:

```text
ModuleNotFoundError: No module named 'torch'
```

the job is using the wrong Python interpreter and should be stopped before debugging DDP or the model.

In [ ]:
# ✏️ RUN THIS

MAX_JOB_ID = submit_and_stream(
    MAX_SLURM_SCRIPT,
    MAX_JOB_NAME,
)

### What success should look like

Early in the log:

```text
=== MAXIMUM DNABERT ===
GPUs: 2
Precision: bf16
Global batch: 64
Local batch/GPU: 32
Trainable percent: 100.00%
```

Then each epoch prints training and validation metrics.

At the end:

```text
=== FINAL REPORT ===
Best AUROC: ...
Training time: ...
Examples/sec: ...
Peak GPU memory: ... / ... GB (...%)
Checkpoint: ...
```

# 11. Load the Saved Maximum-Fine-Tuning Results

In [ ]:
# 🔒 RUN ONLY

RUN_NAME = (
    MAX_CONFIG[
        "run_name"
    ]
)

SUMMARY_PATH = (
    RESULTS_DIR
    / (
        RUN_NAME
        + "_summary.json"
    )
)

HISTORY_PATH = (
    RESULTS_DIR
    / (
        RUN_NAME
        + "_history.csv"
    )
)

PARAMETER_GROUPS_PATH = (
    RESULTS_DIR
    / (
        RUN_NAME
        + "_parameter_groups.csv"
    )
)

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "The maximum DNABERT run "
        "has not produced a summary yet."
    )

with open(
    SUMMARY_PATH
) as handle:
    max_summary = (
        json.load(
            handle
        )
    )

max_history = (
    pd.read_csv(
        HISTORY_PATH
    )
)

parameter_groups = (
    pd.read_csv(
        PARAMETER_GROUPS_PATH
    )
)

pd.Series(
    max_summary,
    name="value",
)

# 12. Parameter Audit

In [ ]:
# 👀 READ — useful model parameters should be fully trainable

parameter_report = pd.DataFrame({
    "metric": [
        "Trainable parameters",
        "Total parameters",
        "Trainable percent",
    ],

    "value": [
        max_summary[
            "trainable_parameters"
        ],

        max_summary[
            "total_parameters"
        ],

        max_summary[
            "trainable_percent"
        ],
    ],
})

parameter_report

In [ ]:
# 👀 READ — trainable parameter count by model component

parameter_plot = (
    parameter_groups[
        parameter_groups[
            "trainable"
        ] == True
    ]
    .sort_values(
        "numel"
    )
)

plt.figure(
    figsize=(9, 6)
)

plt.barh(
    parameter_plot[
        "group"
    ],
    parameter_plot[
        "numel"
    ],
)

plt.xlabel(
    "Trainable parameters"
)

plt.title(
    "Where DNABERT's Trainable Parameters Live"
)

plt.tight_layout()
plt.show()

### ✅ CHECKPOINT — what should you notice?

Most trainable parameters should be inside the **Transformer encoder layers**, not the small classification head.

That is what makes this a true **full fine-tuning** experiment.

# 13. Training Curves

In [ ]:
plt.figure(
    figsize=(7, 4)
)

plt.plot(
    max_history["epoch"],
    max_history[
        "train_loss"
    ],
    marker="o",
    label="Train loss",
)

plt.plot(
    max_history["epoch"],
    max_history[
        "val_loss"
    ],
    marker="o",
    label="Validation loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(
    "Maximum DNABERT — Loss"
)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(7, 4)
)

plt.plot(
    max_history["epoch"],
    max_history[
        "val_auroc"
    ],
    marker="o",
)

plt.xlabel("Epoch")
plt.ylabel("Validation AUROC")
plt.title(
    "Maximum DNABERT — AUROC"
)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(7, 4)
)

plt.plot(
    max_history["epoch"],
    max_history[
        "epoch_seconds"
    ],
    marker="o",
)

plt.xlabel("Epoch")
plt.ylabel("Seconds")
plt.title(
    "Time per Epoch"
)
plt.tight_layout()
plt.show()

# 14. Did We Actually Use the GPU Heavily?

In [ ]:
gpu_report = pd.DataFrame({
    "metric": [
        "GPUs",
        "Precision",
        "Global batch",
        "Local batch / GPU",
        "Training time (s)",
        "Examples / second",
        "Peak GPU memory (GB)",
        "GPU capacity (GB)",
        "Peak memory (%)",
    ],

    "value": [
        max_summary[
            "num_gpus"
        ],

        max_summary[
            "precision"
        ],

        max_summary[
            "global_batch_size"
        ],

        max_summary[
            "local_batch_size"
        ],

        max_summary[
            "training_time_seconds"
        ],

        max_summary[
            "examples_per_second"
        ],

        max_summary[
            "peak_gpu_memory_gb"
        ],

        max_summary[
            "gpu_memory_capacity_gb"
        ],

        max_summary[
            "peak_gpu_memory_percent"
        ],
    ],
})

gpu_report

In [ ]:
# 👀 READ — simple GPU-memory utilization visualization

plt.figure(
    figsize=(6, 4)
)

plt.bar(
    ["Peak allocated memory"],
    [
        max_summary[
            "peak_gpu_memory_percent"
        ]
    ],
)

plt.axhline(
    100,
    linestyle="--",
)

plt.ylabel(
    "% of GPU memory capacity"
)

plt.ylim(
    0,
    105,
)

plt.title(
    "Maximum DNABERT GPU Memory Use"
)

plt.tight_layout()
plt.show()

## How to interpret GPU memory

Peak memory is **not the same thing as GPU compute utilization**.

But it gives us a useful tuning signal.

### If peak memory is low

For example:

```text
< 50%
```

you may be able to increase:

```python
LOCAL_BATCH_SIZE
```

and give each GPU more examples per step.

### If peak memory is very high

For example:

```text
> 90%
```

you are close to the memory limit.

Increasing the batch further could cause an out-of-memory error.

### The goal is not “100% memory at all costs”

The goal is:

> **high useful throughput without crashing or degrading the experiment.**

# 15. Final Model Report

In [ ]:
final_report = pd.DataFrame({
    "metric": [
        "Best validation AUROC",
        "Best epoch",
        "Final accuracy",
        "Final F1",
        "Final AUROC",
        "Final AUPRC",
        "Trainable %",
        "Training time (s)",
        "Examples / second",
    ],

    "value": [
        max_summary[
            "best_val_auroc"
        ],

        max_summary[
            "best_epoch"
        ],

        max_summary[
            "final_val_accuracy"
        ],

        max_summary[
            "final_val_f1"
        ],

        max_summary[
            "final_val_auroc"
        ],

        max_summary[
            "final_val_auprc"
        ],

        max_summary[
            "trainable_percent"
        ],

        max_summary[
            "training_time_seconds"
        ],

        max_summary[
            "examples_per_second"
        ],
    ],
})

final_report

In [ ]:
print(
    "Saved trained model:"
)

print(
    max_summary[
        "checkpoint_path"
    ]
)

# 16. Can You Push the GPU Harder?

Do **not** change many things at once.

The simplest hardware-utilization experiment is the local batch size.

Current:

```python
LOCAL_BATCH_SIZE = 32
```

If the previous run has substantial memory headroom, try:

```python
LOCAL_BATCH_SIZE = 48
```

or:

```python
LOCAL_BATCH_SIZE = 64
```

Then recompute:

```python
GLOBAL_BATCH_SIZE = (
    GPU_COUNT
    * LOCAL_BATCH_SIZE
)
```

and create a **new run name**.

Example:

```python
"run_name": "dnabert_maximum_batch64"
```

### What should you record?

```text
local batch
global batch
peak GPU memory
examples / second
training time
AUROC
```

Increasing batch size is useful only if the extra throughput does not destroy the scientific result.

### 🧠 Student interpretation questions

1. What percentage of the useful model parameters were trainable?
2. Which part of DNABERT contains most of the trainable parameters?
3. How many examples did each GPU process per step?
4. Why do we use BF16 instead of simply converting every model parameter to BF16?
5. What was the peak GPU memory percentage?
6. Could you safely increase the local batch size?
7. Did validation AUROC continue improving for all epochs?
8. Which epoch had the best AUROC?
9. Why might the final epoch not be the best epoch?
10. What is the difference between:
   - maximizing **trainable parameters**, and
   - maximizing **GPU utilization**?

# 17. Notebook 3A Summary

Notebook 3A intentionally does **one thing**:

> **Use the available GPUs to fully fine-tune one DNABERT model.**

The recipe is:

```text
DNA sequences
↓
overlapping 6-mers
↓
pretrained DNABERT-6
↓
remove unused BERT pooler
↓
100% of remaining parameters trainable
↓
BF16 autocast
↓
large local batch on every GPU
↓
DDP gradient synchronization
↓
full fine-tuning
↓
save final trained checkpoint
```

The main hardware measurements are:

```text
training time
examples / second
peak GPU memory
GPU count
```

The main scientific measurements are:

```text
AUROC
AUPRC
accuracy
F1
```

A successful HPC workflow needs **both**:

> fast computation **and** a scientifically useful model.

# ✅ End of Notebook 3A